In [ ]:
# CHALLENGE 1 — Disaster Tweet Classification
# Model: vinai/bertweet-base | Metric: Macro F1

!pip install transformers datasets scikit-learn -q


In [ ]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [ ]:
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
# ── 1. Load Data ──────────────────────────────────────────
train = pd.read_csv("Disaster_with_label.csv")
test  = pd.read_csv("Disaster_no_label.csv")


In [ ]:
# Strip column name spaces
train.columns = train.columns.str.strip()
test.columns  = test.columns.str.strip()

In [ ]:
TEXT_COL  = "Tweet Text"
LABEL_COL = "label"


# Combine all text features for richer signal
train['combined'] = (train['Tweet Text'].fillna('') + ' ' +
                     train['Information Source'].fillna('') + ' ' +
                     train['Information Type'].fillna(''))
test['combined']  = (test['Tweet Text'].fillna('') + ' ' +
                     test['Information Source'].fillna('') + ' ' +
                     test['Information Type'].fillna(''))

TEXT_COL = 'combined'

In [ ]:
# ── 2. Encode Labels ──────────────────────────────────────
# IMPORTANT: labels are "Informative" / "Not Informative"
# We encode for model training, decode back before submission
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['label_enc'] = le.fit_transform(train[LABEL_COL])
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
# e.g. Informative=0, Not Informative=1 (note exact mapping)

Label mapping: {'Informative': np.int64(0), 'Not Informative': np.int64(1)}


In [ ]:
# ── 3. Baseline — TF-IDF + Logistic Regression ───────────
vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X = vec.fit_transform(train[TEXT_COL])
y = train['label_enc']

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='f1_macro')
print(f"Baseline CV Macro F1: {scores.mean():.4f}")


Baseline CV Macro F1: 0.8706


In [ ]:
# Save baseline submission
lr.fit(X, y)
baseline_preds_enc = lr.predict(vec.transform(test[TEXT_COL]))
baseline_preds = le.inverse_transform(baseline_preds_enc)  # back to string labels

test_submit = test.copy()
test_submit[LABEL_COL] = baseline_preds
test_submit.to_csv("c1_baseline_submission.csv", index=False)
print("✅ Baseline submission saved")

✅ Baseline submission saved


In [ ]:
# ── 4. Main Model — BERTweet ──────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MODEL = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=False)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True,
                     padding="max_length", max_length=128)


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


In [ ]:

# Build HF Dataset
train_ds = Dataset.from_pandas(
    train[[TEXT_COL, 'label_enc']].rename(columns={'label_enc': 'label'})
)
train_ds = train_ds.map(tokenize, batched=True)
split = train_ds.train_test_split(test_size=0.1, seed=42)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="./c1_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    seed=42,
    logging_steps=100,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
)
trainer.train()


Map:   0%|          | 0/25933 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2190, training_loss=0.21603426301860373, metrics={'train_runtime': 511.6412, 'train_samples_per_second': 136.848, 'train_steps_per_second': 4.28, 'total_flos': 4605561690785280.0, 'train_loss': 0.21603426301860373, 'epoch': 3.0})

In [ ]:
#  5. Evaluation on Validation Set
val_preds = trainer.predict(split['test'])
val_labels_pred = np.argmax(val_preds.predictions, axis=1)
val_labels_true = split['test']['label']

print("\n" + "="*50)
print("VALIDATION RESULTS — SCREENSHOT THIS")
print("="*50)
print(classification_report(
    val_labels_true, val_labels_pred,
    target_names=le.classes_
))
print(f"Macro F1-Score: {f1_score(val_labels_true, val_labels_pred, average='macro'):.4f}")
print("="*50)



VALIDATION RESULTS — SCREENSHOT THIS
                 precision    recall  f1-score   support

    Informative       0.92      0.93      0.92      1647
Not Informative       0.87      0.86      0.87       947

       accuracy                           0.90      2594
      macro avg       0.90      0.90      0.90      2594
   weighted avg       0.90      0.90      0.90      2594

Macro F1-Score: 0.8955


In [ ]:
# 6. Predict on Test & Save Final Submission
test_ds = Dataset.from_pandas(test[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
final_preds = trainer.predict(test_ds)
final_enc = np.argmax(final_preds.predictions, axis=1)
final_labels = le.inverse_transform(final_enc)  # "Informative" / "Not Informative"

In [ ]:
# label column in no_label.csv exactly as required
test_final = pd.read_csv("Disaster_no_label.csv")
test_final.columns = test_final.columns.str.strip()
test_final['label'] = final_labels

In [ ]:
# Verification
print("\nPredicted label values:", test_final['label'].value_counts().to_dict())
print("Expected: Informative / Not Informative")

test_final.to_csv("Disaster_no_label.csv", index=False)
print("✅ Final submission file saved — Disaster_no_label.csv")


Predicted label values: {'Not Informative': 1147, 'Informative': 853}
Expected: Informative / Not Informative
✅ Final submission file saved — Disaster_no_label.csv


In [ ]:
from google.colab import files
files.download("Disaster_no_label.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
import pandas as pd

# Load final submission file
final = pd.read_csv("Disaster_no_label.csv")
final.columns = final.columns.str.strip()

print("="*50)
print("CHALLENGE 1 — FINAL VERIFICATION")
print("="*50)

# Check 1 — Shape
print(f"\n✅ Total rows: {len(final)} (should be 2000)")

# Check 2 — Columns
print(f"✅ Columns: {final.columns.tolist()}")

# Check 3 — Label values
print(f"\n✅ Label distribution:\n{final['label'].value_counts()}")

# Check 4 — No nulls in label
print(f"\n✅ Null labels: {final['label'].isnull().sum()} (should be 0)")

# Check 5 — Exact label spelling
valid_labels = {'Informative', 'Not Informative'}
unique_labels = set(final['label'].unique())
print(f"\n✅ Unique labels found: {unique_labels}")
print(f"✅ Labels valid: {unique_labels == valid_labels}")

# Check 6 — Show sample
print(f"\n✅ Sample rows:")
print(final[['Tweet Text', 'label']].head(5).to_string())

print("\n" + "="*50)
print("ALL CHECKS PASSED — CHALLENGE 1 COMPLETE! ")
print("="*50)

CHALLENGE 1 — FINAL VERIFICATION

✅ Total rows: 2000 (should be 2000)
✅ Columns: ['Tweet Text', 'Information Source', 'Information Type', 'label']

✅ Label distribution:
label
Not Informative    1147
Informative         853
Name: count, dtype: int64

✅ Null labels: 0 (should be 0)

✅ Unique labels found: {'Informative', 'Not Informative'}
✅ Labels valid: True

✅ Sample rows:
                                                                                                     Tweet Text            label
0                Praying for everyone in Bohol and Cebu esp. for my relatives there. #PrayforBohol #PrayForCebu  Not Informative
1  @tomlinsondisney yep. search for #PrayForVisayas and you'll see the church is destroyed by the earthquake :(  Not Informative
2                  RT @GandangGabiVice: This is really heartbreaking :'( #PrayForVisayas http://t.co/PihNhUmwiW  Not Informative
3                                                   RT @SaklapFriend: God is good all the time. #PrayForVi